# Car Evaluation 

This program predicts car acceptability using two methods:
1. An Expert System (IF-THEN rules)- OpenAI ChatGPT was used to help develop this part of the code.
2. OneR (a machine learning algorithm).

## 1. Load the dataset

In [1]:
import os
import random

ATTRIBUTES = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]
CLASSES = ["unacc", "acc", "good", "vgood"]

# Reads the file and splits each row into a list of values
def load_data(filename):
    with open(filename) as file:
        return [line.strip().split(",") for line in file if line.strip()]

## 2. Expert System (IF-THEN rules)

*The logic for these rules was generated with AI assistance.*

In [2]:
# Figures out the overall price level based on buying and maintenance costs
def get_price(buying, maint):
    if "vhigh" in (buying, maint) and "high" in (buying, maint) or buying == maint == "vhigh": return "vhigh"
    if "vhigh" in (buying, maint): return "high"
    if "high" in (buying, maint): return "high"
    if buying == maint == "med": return "med"
    if "high" in (buying, maint) and "low" in (buying, maint): return "med"
    return "low"

# Figures out the comfort level based on seats, doors, and boot size
def get_comfort(doors, persons, lug_boot):
    if persons == "2": return "low"
    if lug_boot == "big": return "high"
    if lug_boot == "med":
        if persons in ["more", "4"] and doors != "2": return "high"
        return "med"
    if persons == "more" and doors == "2": return "low"
    return "med"

# Figures out the technical level based on comfort and safety
def get_tech(comfort, safety):
    if safety == "low" or comfort == "low": return "low"
    if comfort == "med": return "med" if safety == "med" else "high"
    return "high" if safety == "med" else "vhigh"

# Combines price and tech to get the final car rating
def get_car(price, tech):
    if price == "vhigh" or tech == "low": return "unacc"
    if tech == "med": return "unacc" if price == "high" else "acc"
    if tech == "high": return "good" if price == "low" else "acc"
    return "acc" if price == "high" else "vgood"

# This runs the whole expert system
def expert_predict(row):
    price = get_price(row[0], row[1])
    comfort = get_comfort(row[2], row[3], row[4])
    tech = get_tech(comfort, row[5])
    return get_car(price, tech)

## 3. OneR Algorithm

In [3]:
# This finds the single best attribute and learns the rule from the training data
def train_oner(train_data):
    best_idx, best_err, best_rules = 0, len(train_data) + 1, {}
    
    # Try each of the 6 attributes to see which one is best
    for i in range(6):
        rules = {}
        # For each value, find the most common class
        for val in set(row[i] for row in train_data):
            counts = {}
            for row in train_data:
                if row[i] == val:
                    counts[row[6]] = counts.get(row[6], 0) + 1
            rules[val] = max(counts, key=counts.get)
        
        # this count how many mistakes this attribute makes
        errors = sum(1 for row in train_data if rules[row[i]] != row[6])
        
        # Keep the attribute with the fewest mistakes
        if errors < best_err:
            best_err, best_idx, best_rules = errors, i, rules
            
    return best_idx, best_rules

# this predicts the class using the learned rule
def oner_predict(idx, rules, row):
    return rules.get(row[idx], "unacc")

## 4. Evaluation Metrix

In [4]:

# This calculates the percentage of correct predictions
def get_accuracy(actual, pred):
    return sum(1 for a, p in zip(actual, pred) if a == p) / len(actual)

# Builds and prints a confusion matrix table
def print_matrix(actual, pred):
    matrix = {tc: {pc: 0 for pc in CLASSES} for tc in CLASSES}
    for a, p in zip(actual, pred): matrix[a][p] += 1
    
    # Print the top header
    print(" " * 9, "unacc".ljust(8), "acc".ljust(8), "good".ljust(8), "vgood".ljust(8))
    
    for tc in CLASSES:
        print(tc.ljust(9), end="")
        
        for pc in CLASSES:
            val = matrix[tc][pc]
            # If zero, it becomes an empty string, otherwise the number
            text = "" if val == 0 else str(val)
            print(text.ljust(8), end="")
            
        print()

## 5. Prepare the Data (70/30 Split)

In [5]:
# this loads the data from the folder's parent directory
data = load_data("../data/car.data")  

# Shuffle with a fixed seed so the split is the same every time
random.Random(42).shuffle(data)

# Split 70% for training and 30% for testing
cut = int(len(data) * 0.7)
train_data, test_data = data[:cut], data[cut:]

# This chooses the true answers (column 6) for the test data
actual = [row[6] for row in test_data]

## 6. Run the Expert System

In [6]:
print("\n=== EXPERT SYSTEM ===")

# Get predictions for all test cars
exp_preds = [expert_predict(row) for row in test_data]

# Print accuracy and confusion matrix
print("Accuracy:", round(get_accuracy(actual, exp_preds) * 100, 2), "%")
print_matrix(actual, exp_preds)


=== EXPERT SYSTEM ===
Accuracy: 98.65 %
          unacc    acc      good     vgood   
unacc    353     1                       
acc              116                     
good                     23      1       
vgood            5               20      


## 7. Train and Run One Rule

In [7]:
print("\n=== ONER ===")
# This trains the model to find the best attribute
idx, rules = train_oner(train_data)

# Then gets predictions for all test cars
oner_preds = [oner_predict(idx, rules, row) for row in test_data]

# Prints the learned rules
print("Best Attribute:", ATTRIBUTES[idx])
for val, label in rules.items(): 
    print(f"  if {ATTRIBUTES[idx]}={val} then {label}")

# this Print accuracy and confusion matrix
print("Accuracy:", round(get_accuracy(actual, oner_preds) * 100, 2), "%")
print_matrix(actual, oner_preds)


=== ONER ===
Best Attribute: buying
  if buying=vhigh then unacc
  if buying=med then unacc
  if buying=high then unacc
  if buying=low then unacc
Accuracy: 68.21 %
          unacc    acc      good     vgood   
unacc    354                             
acc      116                             
good     24                              
vgood    25                              
